# Gradient Boosting Mastery: XGBoost and LightGBM in Practice

**What you'll learn**

- Why gradient-boosted trees (GBTs) are the default for tabular prediction — and where linear models and deep learning still win.
- The mechanics: a from-scratch boosting loop that exactly reproduces scikit-learn's `GradientBoostingRegressor`, with staged-fit visualizations.
- XGBoost vs LightGBM: depth-wise vs leaf-wise growth (measured on real trees), speed on the same task, and native categorical/missing handling.
- Which hyperparameters actually matter — a structured one-knob-at-a-time ablation on a fixed dataset, not folklore.
- A sane tuning workflow: early stopping + coarse random search, and an honest look at how modest the tuned-vs-defaults gap usually is.
- Monotonic constraints to encode business logic (price up → P(buy) down), with partial-dependence plots before/after and a ground-truth causal curve.

**Why it matters in interviews**

- "Explain gradient boosting" (at whatever depth the interviewer wants) is a staple of senior ML screens; the from-scratch loop here is the answer.
- Practical-ML rounds probe *judgment*: which knobs you tune first, how you use early stopping, when you stop tuning.
- Case rounds love constraint questions ("the model must respect price monotonicity — what are your options?").
- XGBoost-vs-random-forest and XGBoost-vs-LightGBM comparisons are classic rapid-fire questions.

## Setup

Everything below runs on a single simulated e-commerce dataset with **known ground truth**:
a checkout-stage *purchase propensity* problem. Simulation gives us three things a real dataset
can't: an oracle AUC ceiling (the Bayes-optimal score), a *true causal* price-response curve to
compare partial dependence against, and full control over missingness and confounding.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import lightgbm as lgb
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(42)

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def softplus(z):
    """Numerically stable log(1 + e^z)."""
    return np.log1p(np.exp(-np.abs(z))) + np.maximum(z, 0.0)

## 1. Why gradient-boosted trees dominate tabular problems

The honest version of the story, before any code:

- **Inductive bias fits tabular data.** Tabular features are heterogeneous (dollars, counts,
  ratios, codes), non-smooth, and full of threshold effects and interactions. Axis-aligned
  splits handle all of that natively: no scaling, no encoding gymnastics, robustness to
  irrelevant features and to monotone transformations of inputs.
- **Boosting attacks bias, trees supply variance control.** Each shallow tree is a weak,
  high-bias learner; the stagewise additive fit keeps chipping away at what the ensemble
  still gets wrong. With shrinkage + early stopping you get a bias-variance dial that is
  hard to beat on 10³–10⁷ rows.
- **Versus deep learning:** on mid-sized tabular data (say 1k–1M rows), well-tuned GBTs
  still beat or match well-tuned MLPs/transformers in most published head-to-heads
  (Grinsztajn et al. 2022; Shwartz-Ziv & Armon 2022), while costing 10–100× less to tune.
  Deep learning wins when you have huge data plus unstructured inputs (text, images,
  sequences), need learned embeddings of extreme-cardinality entities, or want multi-task /
  representation sharing. Small-data tabular foundation models (TabPFN-style) are real
  competition below ~10k rows, but GBTs remain the industry default.
- **Versus linear models:** linear/logistic regression wins when n is small relative to p,
  when the signal really is close to linear, when you need coefficients, confidence
  intervals, or a regulator-friendly story — see
  [../01_statistical_foundations/03_regression_the_workhorse.ipynb](../01_statistical_foundations/03_regression_the_workhorse.ipynb).
  GBTs win when there is enough data to learn interactions and curvature you did not
  hand-engineer.

Let's make that concrete: an out-of-the-box bake-off on a realistic dataset.

### The dataset: checkout purchase propensity with known ground truth

One row = one checkout-stage session. The target is *purchase* (base rate ≈ 20%). The
generator builds in the messy features we need later:

- `price` is **confounded** with a *latent* product quality: better products cost more *and*
  convert better. Only a noisy `brand_tier` proxy of quality is observed. The *causal* price
  effect is strictly negative (a softplus curve), but the observational price–conversion
  relationship is flattened and locally non-monotone — exactly the setup where monotonic
  constraints earn their keep (section 6).
- `discount_pct` is **missing not at random**: it is `NaN` on the ~60% of sessions with no
  discount, and those sessions convert less. Missingness itself is signal (section 7).
- `device` (3 levels) and `region_id` (40 levels, random effects) are **categoricals** (section 8).
- Genuinely non-additive structure a linear model can't express without hand-engineering:
  a **U-shaped** `pages_per_session` effect (moderate browsing = engaged, extremes =
  confused or lost), email responsiveness that only pays off on mobile, discounts that
  move new users far more than tenured ones, a "power user" pocket
  (`past_purchases ≥ 2 & sessions ≥ 8`), and price sensitivity that varies with quality.

In [ ]:
N_REGIONS = 40


def price_effect(price):
    """True *causal* effect of price on the log-odds of purchase: smooth, strictly decreasing."""
    return -1.3 * softplus((price - 22.0) / 9.0)


def true_logit(aux, price=None):
    """Ground-truth log-odds for each session; optionally override price (counterfactual)."""
    p = aux["price"] if price is None else price
    device_eff = np.select(
        [aux["device"] == "desktop", aux["device"] == "tablet"], [0.30, -0.15], 0.0
    )
    # price sensitivity varies with quality but stays positive -> causal effect always decreasing
    price_sensitivity = 1.0 - 0.25 * np.clip(aux["quality"], -1.5, 1.5)
    email_eff = aux["email_ctr"] * (1.0 + 5.0 * (aux["device"] == "mobile"))
    discount_base = np.where(
        aux["has_discount"], 0.20 + 0.035 * np.nan_to_num(aux["discount_pct"]), -0.45
    )
    discount_eff = discount_base * np.where(aux["tenure_days"] < 90, 1.8, 0.6)
    power_user = 0.8 * ((aux["past_purchases"] >= 2) & (aux["sessions_30d"] >= 8))
    pages_eff = np.maximum(0.9 - 0.05 * (aux["pages_per_session"] - 5.0) ** 2, -1.2)
    return (
        -2.9
        + price_effect(p) * price_sensitivity
        + 0.8 * aux["quality"]                                     # latent, unobserved
        + 0.45 * np.log1p(aux["sessions_30d"])
        + 0.12 * np.log1p(aux["tenure_days"])
        + email_eff
        + 0.9 * aux["email_ctr"] * np.log1p(aux["sessions_30d"])   # interaction
        + 0.12 * aux["past_purchases"]
        + power_user
        + pages_eff                                                # U-shaped: no linear model ranks this
        + device_eff
        + aux["region_effects"][aux["region_id"]]
        + discount_eff
    )


def make_purchase_data(n, seed):
    """Simulate checkout sessions. Returns (X, y, aux); aux holds latents for counterfactuals."""
    r = np.random.default_rng(seed)
    quality = r.normal(0.0, 1.0, n)                          # latent product quality
    brand_tier = np.clip(np.rint(3 + 1.1 * quality + r.normal(0, 0.9, n)), 1, 5)
    price = np.clip(28 + 9 * quality + r.normal(0, 5, n), 5, None).round(2)
    tenure_days = r.gamma(2.0, 180.0, n).round(0)
    sessions_30d = r.poisson(np.exp(1.0 + 0.5 * r.normal(0, 1, n)))
    pages_per_session = (1 + r.gamma(2.0, 1.5, n)).round(1)
    email_ctr = r.beta(1.5, 12.0, n).round(4)
    past_purchases = r.poisson(0.5 + np.minimum(tenure_days, 720) / 400.0)
    device = r.choice(np.array(["mobile", "desktop", "tablet"]), size=n, p=[0.55, 0.35, 0.10])
    region_id = r.integers(0, N_REGIONS, n)
    region_effects = r.normal(0.0, 0.4, N_REGIONS)
    has_discount = r.random(n) < 0.40
    discount_pct = np.where(has_discount, np.round(r.uniform(5, 30, n)), np.nan)

    aux = dict(
        quality=quality, price=price, tenure_days=tenure_days, sessions_30d=sessions_30d,
        pages_per_session=pages_per_session, email_ctr=email_ctr,
        past_purchases=past_purchases, device=device,
        region_id=region_id, region_effects=region_effects,
        has_discount=has_discount, discount_pct=discount_pct,
    )
    y = (r.random(n) < sigmoid(true_logit(aux))).astype(int)

    X = pd.DataFrame({
        "price": price,
        "brand_tier": brand_tier,
        "tenure_days": tenure_days,
        "sessions_30d": sessions_30d,
        "pages_per_session": pages_per_session,
        "email_ctr": email_ctr,
        "past_purchases": past_purchases,
        "discount_pct": discount_pct,
        "device": pd.Categorical(device),
        "region_id": pd.Categorical(region_id),
    })
    return X, y, aux


X_all, y_all, aux = make_purchase_data(60_000, seed=42)
FEATURES = list(X_all.columns)

# 60/20/20 train / valid / test split
perm = rng.permutation(len(X_all))
i_tr, i_va, i_te = perm[:36_000], perm[36_000:48_000], perm[48_000:]
X_train, X_valid, X_test = X_all.iloc[i_tr], X_all.iloc[i_va], X_all.iloc[i_te]
y_train, y_valid, y_test = y_all[i_tr], y_all[i_va], y_all[i_te]

oracle_auc = roc_auc_score(y_all, true_logit(aux))
print(f"rows: {len(X_all):,}   base rate: {y_all.mean():.3f}")
print(f"oracle AUC (Bayes-optimal, scoring with the true logit): {oracle_auc:.4f}")
print(f"discount_pct missing: {X_all['discount_pct'].isna().mean():.1%}")
X_all.head()

### Bake-off: logistic regression vs random forest vs LightGBM, out of the box

Same split for everyone. The linear model needs a real preprocessing pipeline (impute +
missing indicator + scale + one-hot); the random forest needs categorical codes; LightGBM
consumes the raw frame — `NaN`s, categoricals and all.

In [ ]:
results = []

# --- Logistic regression: full preprocessing pipeline ---
num_cols = [c for c in FEATURES if c not in ("device", "region_id")]
logreg = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", StandardScaler()),
        ]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["device", "region_id"]),
    ])),
    ("clf", LogisticRegression(max_iter=2000)),
])
t0 = time.perf_counter()
logreg.fit(X_train, y_train)
results.append({
    "model": "logistic regression (impute+scale+one-hot)",
    "fit_s": time.perf_counter() - t0,
    "test_auc": roc_auc_score(y_test, logreg.predict_proba(X_test)[:, 1]),
})

# --- Random forest: needs categorical codes; handles NaN natively since sklearn 1.4 ---
def to_codes(X):
    X = X.copy()
    for c in ("device", "region_id"):
        X[c] = X[c].cat.codes
    return X

t0 = time.perf_counter()
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=10, n_jobs=-1, random_state=42)
rf.fit(to_codes(X_train), y_train)
results.append({
    "model": "random forest (300 trees)",
    "fit_s": time.perf_counter() - t0,
    "test_auc": roc_auc_score(y_test, rf.predict_proba(to_codes(X_test))[:, 1]),
})

# --- LightGBM: raw frame, early stopping on the validation set ---
t0 = time.perf_counter()
lgbm = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, random_state=42, verbose=-1)
lgbm.fit(X_train, y_train, eval_X=X_valid, eval_y=y_valid, eval_metric="auc",
         callbacks=[lgb.early_stopping(50, verbose=False, first_metric_only=True)])
results.append({
    "model": f"LightGBM (early-stopped @ {lgbm.best_iteration_} trees)",
    "fit_s": time.perf_counter() - t0,
    "test_auc": roc_auc_score(y_test, lgbm.predict_proba(X_test)[:, 1]),
})

bakeoff = pd.DataFrame(results).assign(
    gap_to_oracle=lambda d: oracle_auc - d["test_auc"]
).round({"fit_s": 2, "test_auc": 4, "gap_to_oracle": 4})
print(f"oracle AUC ceiling: {oracle_auc:.4f}")
bakeoff

Read this table like a senior candidate, not a leaderboard junkie:

- LightGBM wins, and got there with **zero preprocessing** — the pipeline work that the
  linear model *required* (imputation, indicators, scaling, one-hot) is where real-world
  bugs and leakage live.
- The AUC gap over logistic regression looks small in absolute terms (~0.01–0.02 here).
  That is typical for tabular problems — and at the scale of a large product, a point of
  AUC is real money. But if a stakeholder needs *coefficients*, the linear model's gap may
  be a price worth paying.
- The random forest trails both here: bagging reduces variance only, and ordinal-coded
  categoricals hurt it; boosting also attacks bias and self-sizes its capacity via early
  stopping. RF's appeal is robustness with near-zero tuning, not peak accuracy.
- Nobody reaches the oracle ceiling: irreducible noise plus the *unobserved* quality
  confounder cap what any model can do. Knowing the ceiling exists (even when you can't
  compute it) is what stops you from burning weeks tuning for noise.

## 2. Mechanics: gradient boosting from scratch

Three sentences of theory, then code.

Gradient boosting is **gradient descent in function space**: maintain an additive model
$F_m(x) = F_0 + \nu \sum_{k \le m} h_k(x)$, and at each step fit a small tree $h_m$ to the
**negative gradient of the loss** evaluated at the current predictions —
$r_i = -\partial L(y_i, F(x_i)) / \partial F(x_i)$ — then take a small step $\nu$ (the
learning rate) in that direction. For squared loss the negative gradient *is* the residual
$y_i - F(x_i)$, which is why "fit trees to residuals" is the correct one-line summary; for
log-loss it is $y_i - p_i$, and any differentiable loss works. XGBoost refines the step by
also using second derivatives (a Newton step per leaf) plus explicit regularization on tree
structure — more on that in the interview corner.

We boost depth-2 trees on a 1D regression problem so every stage is visible.

In [ ]:
rng_1d = np.random.default_rng(7)


def f_true(x):
    return 2.0 * np.sin(1.5 * x) + 0.5 * x + 1.5 * (x > 4.0)


n_1d = 300
x_obs = np.sort(rng_1d.uniform(0, 6, n_1d))
y_obs = f_true(x_obs) + rng_1d.normal(0, 0.4, n_1d)
x_new = np.sort(rng_1d.uniform(0, 6, n_1d))
y_new = f_true(x_new) + rng_1d.normal(0, 0.4, n_1d)
x_grid = np.linspace(0, 6, 400)


def boost_from_scratch(x, y, n_trees, lr, max_depth=2):
    """Plain gradient boosting for squared loss. Returns (init, trees)."""
    X = x.reshape(-1, 1)
    F = np.full(len(y), y.mean())
    trees = []
    for _ in range(n_trees):
        residual = y - F                       # negative gradient of 1/2 (y - F)^2
        tree = DecisionTreeRegressor(max_depth=max_depth, random_state=0)
        tree.fit(X, residual)
        F = F + lr * tree.predict(X)           # shrunken step in function space
        trees.append(tree)
    return y.mean(), trees


def boost_predict(init, trees, lr, x, n_use=None):
    X = x.reshape(-1, 1)
    use = trees if n_use is None else trees[:n_use]
    out = np.full(len(x), init)
    for t in use:
        out = out + lr * t.predict(X)
    return out


init, trees = boost_from_scratch(x_obs, y_obs, n_trees=150, lr=0.1)

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)
for ax, m in zip(axes.ravel(), [1, 5, 25, 150]):
    ax.scatter(x_obs, y_obs, s=6, alpha=0.3, color="gray", label="train data")
    ax.plot(x_grid, f_true(x_grid), "k--", lw=1.2, label="true f(x)")
    ax.plot(x_grid, boost_predict(init, trees, 0.1, x_grid, n_use=m), color="C3", lw=1.8,
            label=f"$F_{{{m}}}$")
    ax.set_title(f"after {m} tree{'s' if m > 1 else ''}")
    ax.legend(loc="upper left", fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("x")
for ax in axes[:, 0]:
    ax.set_ylabel("y")
fig.suptitle("Staged fits: each depth-2 tree corrects the current residuals (lr = 0.1)", y=1.02)
fig.tight_layout()
plt.show()

The first tree is a crude 4-step staircase; by 25 trees the shape is there; by 150 the fit
hugs the true curve, including the discontinuity at $x=4$ — the kind of jump a spline or a
neural net would smooth over.

### The learning-rate / n_estimators tradeoff

Shrinkage is regularization: a small $\nu$ means each tree commits less to the *current*
residuals, leaving room for later trees to correct. The cost is more trees to reach the
same training loss; the benefit is a better test-loss minimum and a much flatter,
more forgiving overfitting curve.

In [ ]:
def mse_curves(lr, n_trees=300):
    X_tr, X_te = x_obs.reshape(-1, 1), x_new.reshape(-1, 1)
    F_tr = np.full(len(y_obs), y_obs.mean())
    F_te = np.full(len(y_new), y_obs.mean())
    tr, te = [], []
    for _ in range(n_trees):
        tree = DecisionTreeRegressor(max_depth=2, random_state=0)
        tree.fit(X_tr, y_obs - F_tr)
        F_tr = F_tr + lr * tree.predict(X_tr)
        F_te = F_te + lr * tree.predict(X_te)
        tr.append(np.mean((y_obs - F_tr) ** 2))
        te.append(np.mean((y_new - F_te) ** 2))
    return np.array(tr), np.array(te)


fig, ax = plt.subplots(figsize=(8, 4.2))
for lr_val, color in [(1.0, "C3"), (0.1, "C0"), (0.03, "C2")]:
    tr, te = mse_curves(lr_val)
    ax.plot(te, color=color, lw=1.6, label=f"test,  lr={lr_val}")
    ax.plot(tr, color=color, lw=1.0, ls=":", alpha=0.7, label=f"train, lr={lr_val}")
    print(f"lr={lr_val:<5} best test MSE {te.min():.4f} at tree {te.argmin() + 1:>3}   "
          f"final test MSE {te[-1]:.4f}")
ax.axhline(0.4 ** 2, color="k", ls="--", lw=1, label="noise floor $\\sigma^2$")
ax.set(xlabel="number of trees", ylabel="MSE", ylim=(0, 1.2),
       title="Shrinkage as regularization: small lr reaches a lower test minimum and overfits slowly")
ax.legend(ncol=2, fontsize=8)
plt.show()

`lr=1.0` races to a worse minimum and then climbs as it fits noise; `lr=0.1` and `lr=0.03`
both settle near the noise floor $\sigma^2 = 0.16$, and their test curves stay flat for
hundreds of trees — which is exactly why the standard recipe is *"pick a smallish learning
rate, make `n_estimators` effectively infinite, and let early stopping choose the count."*

### Sanity check: our loop *is* scikit-learn's

Same learning rate, depth, and tree count should give bit-for-bit identical predictions to
`GradientBoostingRegressor` (its default `friedman_mse` split criterion is equivalent to
squared error for regression).

In [ ]:
gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=2, random_state=0)
gbr.fit(x_obs.reshape(-1, 1), y_obs)
ours = boost_predict(init, trees, 0.1, x_grid, n_use=100)
print("max |from-scratch − sklearn| on a dense grid:",
      np.abs(ours - gbr.predict(x_grid.reshape(-1, 1))).max())

Zero. Gradient boosting is not magic — it is 15 lines of NumPy plus a tree learner.
Everything XGBoost and LightGBM add is about *speed* (histograms, sparsity handling,
parallelism) and *regularization* (second-order leaf weights, L1/L2, smarter growth).

## 3. XGBoost vs LightGBM

Both are histogram-based: continuous features are bucketed into ≤255 bins, and split search
scans bin boundaries — that is the single biggest speed trick. The headline differences:

| | XGBoost | LightGBM |
|---|---|---|
| default growth | **depth-wise** (level by level, balanced trees; `grow_policy="lossguide"` opts into leaf-wise) | **leaf-wise** (always split the leaf with the largest loss reduction) |
| capacity knob | `max_depth` | `num_leaves` (with `max_depth` as a guard) |
| categorical | native (`enable_categorical=True`, category dtype) | native (category dtype), sorted-by-gradient-statistics splits |
| extras | second-order leaf weights, `gamma` split penalty | GOSS sampling, exclusive feature bundling (EFB) |

Leaf-wise growth spends its leaf budget where the loss says it matters, so with the same
number of leaves it builds **deeper, asymmetric** trees — usually more accurate per leaf,
and easier to overfit with (hence `num_leaves` is LightGBM's overfit knob, section 4).
Let's measure both claims: wall-clock on a bigger sample, then the actual tree shapes.

In [ ]:
X_big, y_big, _ = make_purchase_data(250_000, seed=123)
Xb_train, Xb_test = X_big.iloc[:200_000], X_big.iloc[200_000:]
yb_train, yb_test = y_big[:200_000], y_big[200_000:]

speed_rows = []
for name, model in [
    ("LightGBM (leaf-wise, 63 leaves)",
     lgb.LGBMClassifier(n_estimators=400, num_leaves=63, learning_rate=0.1,
                        random_state=42, verbose=-1)),
    ("XGBoost (depth-wise, max_depth=6)",
     xgb.XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.1,
                       tree_method="hist", enable_categorical=True, random_state=42)),
]:
    t0 = time.perf_counter()
    model.fit(Xb_train, yb_train)
    fit_s = time.perf_counter() - t0
    t0 = time.perf_counter()
    proba = model.predict_proba(Xb_test)[:, 1]
    pred_s = time.perf_counter() - t0
    speed_rows.append({"model": name, "fit_s": fit_s, "predict_s": pred_s,
                       "test_auc": roc_auc_score(yb_test, proba)})

pd.DataFrame(speed_rows).round({"fit_s": 2, "predict_s": 2, "test_auc": 4})

Same task, same tree budget, near-identical AUC — LightGBM is meaningfully faster to fit
(typically 1.5–3× on CPU; exact ratios depend on cores and data shape — this box is a small
shared container, so treat the wall-clock as directional). At interview depth: LightGBM's
edge comes from leaf-wise growth doing fewer wasted splits, GOSS-style sampling options,
feature bundling for sparse data, and a leaner histogram implementation. XGBoost's edge is
maturity of the distributed/GPU story and some extra regularization controls. On most
tabular problems **they converge to the same accuracy** once tuned; choice is workflow, not
accuracy.

### Look at the trees they actually grow

In [ ]:
lgb_anatomy = lgb.LGBMClassifier(n_estimators=60, num_leaves=63, learning_rate=0.1,
                                 random_state=42, verbose=-1)
lgb_anatomy.fit(X_train, y_train)
ltree = lgb_anatomy.booster_.trees_to_dataframe()
lgb_leaves = ltree[ltree["split_feature"].isna()].copy()
lgb_leaves["depth"] = lgb_leaves["node_depth"] - 1        # edges from root

xgb_anatomy = xgb.XGBClassifier(n_estimators=60, max_depth=6, learning_rate=0.1,
                                tree_method="hist", enable_categorical=True, random_state=42)
xgb_anatomy.fit(X_train, y_train)
xtree = xgb_anatomy.get_booster().trees_to_dataframe()


def xgb_leaf_depths(tdf):
    """Depth of every leaf, via BFS over the child pointers (Yes/No columns)."""
    depths = []
    for _, tree_df in tdf.groupby("Tree"):
        kids = tree_df.set_index("ID")[["Yes", "No", "Feature"]]
        frontier = [(tree_df["ID"].iloc[0], 0)]
        while frontier:
            node, d = frontier.pop()
            row = kids.loc[node]
            if row["Feature"] == "Leaf":
                depths.append(d)
            else:
                frontier += [(row["Yes"], d + 1), (row["No"], d + 1)]
    return np.array(depths)


xgb_depths = xgb_leaf_depths(xtree)
lgb_per_tree = lgb_leaves.groupby("tree_index").size()
xgb_per_tree = xtree[xtree["Feature"] == "Leaf"].groupby("Tree").size()

print(f"LightGBM: {lgb_per_tree.mean():.1f} leaves/tree, "
      f"leaf depth mean {lgb_leaves['depth'].mean():.1f}, max {lgb_leaves['depth'].max()}")
print(f"XGBoost : {xgb_per_tree.mean():.1f} leaves/tree, "
      f"leaf depth mean {xgb_depths.mean():.1f}, max {xgb_depths.max()}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
axes[0].hist(lgb_leaves["depth"], bins=np.arange(0.5, lgb_leaves["depth"].max() + 1.5),
             color="C0", density=True)
axes[0].set(title="LightGBM leaf depths (63-leaf budget, leaf-wise)",
            xlabel="leaf depth", ylabel="fraction of leaves")
axes[1].hist(xgb_depths, bins=np.arange(0.5, xgb_depths.max() + 1.5), color="C1", density=True)
axes[1].set(title="XGBoost leaf depths (max_depth=6, depth-wise)", xlabel="leaf depth")
fig.suptitle("Leaf-wise growth builds deep, asymmetric trees; depth-wise builds balanced ones", y=1.04)
fig.tight_layout()
plt.show()

The histograms are the picture to carry into interviews: XGBoost's leaves pile up at its
depth cap (balanced trees), while LightGBM — given the *same order of leaf count* — grows
lopsided trees where the loss demands it: similar mean leaf depth, but maximum depth ~3×
XGBoost's cap (19 vs 6 here). Deeper paths mean higher-order interactions captured with
the same leaf budget, and also why an unconstrained `num_leaves` can quietly build 20-deep
branches that memorize noise.

## 4. The parameters that matter

The knobs, in the order they matter, for LightGBM (XGBoost equivalents in parentheses):

1. **`learning_rate` + `n_estimators` via early stopping** — the capacity schedule. Not
   really "tuned": set lr ∈ [0.03, 0.1], cap trees high, early-stop.
2. **`num_leaves`** (`max_depth`) — *the* overfit knob: model complexity per tree.
3. **`min_child_samples`** (`min_child_weight`) — minimum data per leaf; the brake on
   memorizing tiny noise pockets.
4. **`subsample` + `colsample_bytree`** — stochasticity; decorrelates trees, speeds up fits.
5. **`reg_alpha` / `reg_lambda`** (L1/L2 on leaf values) — usually second-order refinements.

A structured ablation beats an exhaustive grid for *understanding*: fix a sensible baseline,
move one knob at a time, watch train AUC, valid AUC, and the early-stopped tree count.

In [ ]:
BASE = dict(n_estimators=4000, learning_rate=0.05, num_leaves=31, min_child_samples=20,
            subsample=1.0, subsample_freq=1, colsample_bytree=1.0,
            reg_alpha=0.0, reg_lambda=0.0, random_state=42, verbose=-1)


def fit_eval(knob, value, **overrides):
    """Fit LightGBM with early stopping; return one ablation row (+ the model)."""
    model = lgb.LGBMClassifier(**{**BASE, **overrides})
    t0 = time.perf_counter()
    model.fit(X_train, y_train, eval_X=X_valid, eval_y=y_valid, eval_metric="auc",
              callbacks=[lgb.early_stopping(50, verbose=False, first_metric_only=True)])
    row = {
        "knob": knob, "value": value, "best_iter": model.best_iteration_,
        "train_auc": roc_auc_score(y_train, model.predict_proba(X_train)[:, 1]),
        "valid_auc": roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1]),
        "fit_s": time.perf_counter() - t0,
    }
    row["train_valid_gap"] = row["train_auc"] - row["valid_auc"]
    return row, model


baseline_row, baseline_model = fit_eval("baseline", "-")
print(pd.DataFrame([baseline_row]).round(4).to_string(index=False))

### Knob 1: learning rate (with early stopping choosing the tree count)

In [ ]:
lr_rows, lr_curves = [], {}
for lr_val in [0.3, 0.1, 0.05, 0.01]:
    row, model = fit_eval("learning_rate", lr_val, learning_rate=lr_val)
    lr_rows.append(row)
    lr_curves[lr_val] = model.evals_result_["valid_0"]["auc"]

lr_table = pd.DataFrame(lr_rows).round(4)
print(lr_table.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.2))
for lr_val, curve in lr_curves.items():
    ax.plot(np.arange(1, len(curve) + 1), curve, lw=1.5, label=f"lr={lr_val}")
ax.set(xlabel="boosting round", ylabel="validation AUC", xscale="log",
       title="learning_rate × n_estimators tradeoff (early stopping picks the endpoint)")
ax.legend()
plt.show()

All learning rates plateau at essentially the same validation AUC — they trade **trees for
step size** (lr=0.3 stops after tens of trees, lr=0.01 needs thousands). The practical rule:
lr in the 0.03–0.1 range buys you the smooth, forgiving part of the curve at acceptable
cost; below that you pay compute for nothing on a dataset this size. Tiny learning rates
only earn their keep on very large datasets where the extra trees keep finding signal.

### Knob 2 and 3: `num_leaves` (complexity) and `min_child_samples` (leaf evidence)

In [ ]:
leaves_rows = [fit_eval("num_leaves", v, num_leaves=v)[0] for v in [7, 15, 31, 63, 127, 255]]
mcs_rows = [fit_eval("min_child_samples", v, min_child_samples=v)[0] for v in [1, 5, 20, 100, 500]]
leaves_df, mcs_df = pd.DataFrame(leaves_rows), pd.DataFrame(mcs_rows)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, d, xcol in [(axes[0], leaves_df, "num_leaves"), (axes[1], mcs_df, "min_child_samples")]:
    ax.plot(d["value"], d["train_auc"], "o-", label="train AUC")
    ax.plot(d["value"], d["valid_auc"], "o-", label="valid AUC")
    ax.set(xscale="log", xlabel=xcol, ylabel="AUC", title=f"{xcol} (early-stopped)")
    ax.legend(fontsize=8)
fig.suptitle("The overfit knobs: capacity per tree vs evidence per leaf", y=1.04)
fig.tight_layout()
plt.show()
print(leaves_df.round(4).to_string(index=False))
print()
print(mcs_df.round(4).to_string(index=False))

Classic pattern: train AUC rises monotonically with `num_leaves` while validation AUC
*falls* — on this dataset the smallest tree we tried (7 leaves) is actually the best, and
even with early stopping as a safety net, extra capacity only buys a bigger train/valid
gap, not a better model. `min_child_samples` barely moves validation AUC here
(60k clean rows); on small or noisy data it becomes a first-order knob. Note *how* early
stopping compensates: bigger trees → fewer boosting rounds.

### Knobs 4 and 5: stochasticity and L1/L2, plus the combined ablation table

In [ ]:
stoch_rows = [
    fit_eval("subsample", v, subsample=v)[0] for v in [0.5, 0.8]
] + [
    fit_eval("colsample_bytree", v, colsample_bytree=v)[0] for v in [0.5, 0.8]
]
reg_rows = [
    fit_eval("reg_lambda", v, reg_lambda=v)[0] for v in [1.0, 10.0, 100.0]
] + [
    fit_eval("reg_alpha", v, reg_alpha=v)[0] for v in [1.0, 10.0]
]

ablation = pd.concat([
    pd.DataFrame([baseline_row]), lr_table, leaves_df, mcs_df,
    pd.DataFrame(stoch_rows), pd.DataFrame(reg_rows),
], ignore_index=True)
ablation["delta_vs_baseline"] = ablation["valid_auc"] - baseline_row["valid_auc"]
ablation.round(4)

What the ablation says on *this* dataset (a pattern that generalizes surprisingly well):

- **No single knob moves validation AUC by even a hundredth** once early stopping is in
  place — the biggest move in the whole table is cranking `num_leaves` to 255 (−0.008);
  most rows sit within ±0.004 of baseline. Early stopping is doing most of the
  regularization work.
- The knobs' *main observable effect* is on the train/valid gap and the tree count —
  i.e., on how hard early stopping has to work — not on the achievable score.
- Stochasticity (`subsample`, `colsample_bytree` < 1) is cheap, mild regularization: here
  both *nudge validation AUC up* a few thousandths while making each tree cheaper to fit.
  L1/L2 are near no-ops here (|Δ| ≤ 0.0013) and typically only matter with many weak/noisy
  features or very small leaves.
- Corollary for real work: get the *data and validation design* right first
  ([02_leakage_validation.ipynb](02_leakage_validation.ipynb)); hyperparameters are the
  last 1%, and the next section shows exactly how much that 1% is worth.

## 5. A sane tuning workflow

The workflow that survives contact with production:

1. **Baseline**: library defaults. Costs one fit; this is your floor.
2. **Defaults + early stopping**: lr ≈ 0.05, huge `n_estimators`, stop on a validation set.
3. **Coarse random search** over the 3–4 knobs that matter (`num_leaves`,
   `min_child_samples`, `subsample`/`colsample`, `reg_lambda`), ~25–50 trials, each fit
   early-stopped. Random beats grid at equal budget because it doesn't waste trials on
   unimportant dimensions (Bergstra & Bengio 2012).
4. **Stop.** Report the tuned-vs-step-2 gap honestly; it is usually small.

In [ ]:
# Step 1 + 2: the two cheap baselines
defaults = lgb.LGBMClassifier(random_state=42, verbose=-1)
defaults.fit(X_train, y_train)
auc_defaults = roc_auc_score(y_test, defaults.predict_proba(X_test)[:, 1])
auc_es = roc_auc_score(y_test, baseline_model.predict_proba(X_test)[:, 1])  # section-4 baseline

# Step 3: coarse random search, 30 trials
search_rng = np.random.default_rng(0)
trials = []
t0 = time.perf_counter()
for trial in range(30):
    params = dict(
        num_leaves=int(2 ** search_rng.uniform(3, 8)),                # 8 .. 256, log scale
        min_child_samples=int(10 ** search_rng.uniform(0.7, 2.5)),    # 5 .. ~316, log scale
        subsample=search_rng.uniform(0.6, 1.0),
        colsample_bytree=search_rng.uniform(0.6, 1.0),
        reg_lambda=10 ** search_rng.uniform(-3, 1.5),                 # 0.001 .. ~32
    )
    row, model = fit_eval("search", trial, n_estimators=1500, **params)
    row.update(params)
    row["test_auc"] = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])  # logged, not used for selection
    trials.append(row)
search_s = time.perf_counter() - t0

search = pd.DataFrame(trials).sort_values("valid_auc", ascending=False)
best = search.iloc[0]
print(f"30 early-stopped trials in {search_s:.0f}s")
print("best trial by valid AUC:")
print(best[["num_leaves", "min_child_samples", "subsample", "colsample_bytree",
            "reg_lambda", "best_iter", "valid_auc"]].astype(float).round(4).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
best_so_far = search.sort_index()["valid_auc"].cummax()
ax.plot(np.arange(1, 31), best_so_far, "o-", ms=4)
ax.axhline(baseline_row["valid_auc"], color="C3", ls="--", lw=1,
           label=f"defaults + early stopping ({baseline_row['valid_auc']:.4f})")
ax.set(xlabel="random-search trial", ylabel="best validation AUC so far",
       title="Diminishing returns: most of the tuning gain arrives in the first few trials")
ax.legend()
plt.show()

In [ ]:
tuning_summary = pd.DataFrame({
    "config": ["library defaults (100 trees, lr=0.1)",
               "sensible defaults + early stopping",
               "random search, 30 trials (selected on valid)"],
    "test_auc": [auc_defaults, auc_es, best["test_auc"]],
}).assign(gain_vs_defaults=lambda d: d["test_auc"] - d["test_auc"].iloc[0]).round(4)
tuning_summary

The pattern to internalize (and to say out loud in interviews): **the gap between
"sensible defaults + early stopping" and "30 trials of random search" is a few thousandths
of AUC.** The best-so-far curve is nearly flat after the first handful of trials. Tuning
is real but it is the *last* lever: new features, more data, fixing leakage, and a correct
validation split each dwarf it. Budget accordingly — and when someone reports a big tuning
win, first suspect their validation design
([02_leakage_validation.ipynb](02_leakage_validation.ipynb)) before their search strategy.

## 6. Monotonic constraints: encoding business logic

The business rule: *raising the price of an item must not raise its predicted purchase
probability*. Our generator guarantees the **causal** price effect is strictly negative.
But the model sees *observational* data where price is confounded with latent quality
(better products cost more *and* convert better, and `brand_tier` is only a noisy proxy).
An unconstrained model therefore learns a price response that is too flat and locally
**increasing** — if pricing simulations or "what-if" tools consume this model, it will
happily recommend price hikes.

We compare three curves: the unconstrained model's partial dependence, the constrained
model's, and the *true causal* dose-response curve (computable only because we simulated).

In [ ]:
price_grid = np.arange(12.0, 56.0, 1.0)
pdp_sample = X_test.iloc[:3000]


def pdp_price(model, X, grid):
    """Manual partial dependence: set price to v for everyone, average predicted probability."""
    vals = []
    for v in grid:
        X_mod = X.copy()
        X_mod["price"] = v
        vals.append(model.predict_proba(X_mod)[:, 1].mean())
    return np.array(vals)


def frac_increasing(curve):
    steps = np.diff(curve)
    return float((steps > 0).mean())


# true causal dose-response, averaged over the population
causal_curve = np.array([sigmoid(true_logit(aux, price=v)).mean() for v in price_grid])

pdp_free = pdp_price(baseline_model, pdp_sample, price_grid)
print(f"unconstrained PDP: {frac_increasing(pdp_free):.0%} of grid steps are INCREASING "
      f"(max up-step {np.diff(pdp_free).max():+.4f})")

In [ ]:
mono_lgb = lgb.LGBMClassifier(**BASE, monotone_constraints=[-1 if c == "price" else 0 for c in FEATURES])
mono_lgb.fit(X_train, y_train, eval_X=X_valid, eval_y=y_valid, eval_metric="auc",
             callbacks=[lgb.early_stopping(50, verbose=False, first_metric_only=True)])

# XGBoost spells the same constraint as a dict keyed by feature name
mono_xgb = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=6,
                             tree_method="hist", enable_categorical=True,
                             monotone_constraints={"price": -1},
                             eval_metric="auc", early_stopping_rounds=50, random_state=42)
mono_xgb.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

pdp_mono = pdp_price(mono_lgb, pdp_sample, price_grid)
print(f"constrained PDP:   {frac_increasing(pdp_mono):.0%} of grid steps are increasing "
      f"(max up-step {np.diff(pdp_mono).max():+.6f})")

mono_table = pd.DataFrame({
    "model": ["LightGBM unconstrained", "LightGBM price ↓ constraint", "XGBoost price ↓ constraint"],
    "test_auc": [
        roc_auc_score(y_test, baseline_model.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_test, mono_lgb.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_test, mono_xgb.predict_proba(X_test)[:, 1]),
    ],
}).round(4)
print(mono_table.to_string(index=False))

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(price_grid, pdp_free, "o-", ms=3, color="C3", label="unconstrained PDP")
ax.plot(price_grid, pdp_mono, "o-", ms=3, color="C0", label="monotone-constrained PDP")
ax.plot(price_grid, causal_curve, "k--", lw=1.5, label="TRUE causal dose-response (oracle)")
ax.set(xlabel="price ($)", ylabel="mean predicted P(purchase)",
       title="Partial dependence on price: the constraint removes the confounding-driven up-wiggles")
ax.legend()
plt.show()

Three senior-level observations:

1. **The constraint is nearly free predictively** — the test AUCs above differ by a couple
   of thousandths, noise-level money for *guaranteed* sane behavior in every pricing
   what-if. When the constrained direction matches the true signal, the constraint can even
   act as a helpful regularizer; a large accuracy drop after constraining is itself a
   diagnostic that your "business logic" contradicts the data.
2. **The constraint fixes the *shape*, not the *magnitude*.** Both model curves are far
   flatter than the true causal curve, because price still proxies unobserved quality and
   the model has no way to know better. A monotone model is safe for *directional*
   what-ifs, but for "how much would demand drop if we raised prices 10%?" you need causal
   identification — see
   [../03_causal_inference/01_potential_outcomes_dags.ipynb](../03_causal_inference/01_potential_outcomes_dags.ipynb).
3. Under the hood the implementation is simple: during tree growth, any split on `price`
   whose child predictions would violate the declared direction is clipped/rejected, and
   bounds propagate down the subtree. Cheap at train time, exact at predict time.

## 7. Native missing-value handling

Both libraries treat `NaN` as a first-class value: at every split, the missing rows are
routed to whichever side reduces loss more (a *learned default direction*). This matters
most when missingness is **informative** (MNAR) — as with `discount_pct`, which is missing
precisely for sessions that got no discount and convert less.

In [ ]:
median_disc = X_train["discount_pct"].median()


def impute_median(X):
    X = X.copy()
    X["discount_pct"] = X["discount_pct"].fillna(median_disc)
    return X


def impute_flag(X):
    X_imp = impute_median(X)
    X_imp["discount_was_missing"] = X["discount_pct"].isna().astype(int)
    return X_imp


missing_rows = []
for strategy, f in [("native NaN handling", lambda X: X),
                    ("median imputation", impute_median),
                    ("median imputation + missing flag", impute_flag)]:
    model = lgb.LGBMClassifier(**BASE)
    model.fit(f(X_train), y_train, eval_X=f(X_valid), eval_y=y_valid, eval_metric="auc",
              callbacks=[lgb.early_stopping(50, verbose=False, first_metric_only=True)])
    missing_rows.append({
        "strategy": strategy,
        "test_auc": roc_auc_score(y_test, model.predict_proba(f(X_test))[:, 1]),
    })
print(pd.DataFrame(missing_rows).round(4).to_string(index=False))

# where do the trees send the missing rows?
tdf = baseline_model.booster_.trees_to_dataframe()
disc_splits = tdf[tdf["split_feature"] == "discount_pct"]
print(f"\nsplits on discount_pct in the baseline model: {len(disc_splits)}")
print("learned default direction for missing values:")
print(disc_splits["missing_direction"].value_counts().to_string())

Takeaways:

- **Plain median imputation is the worst option**: it collapses "no discount" onto real
  mid-range discounts and destroys the MNAR signal.
- **Native handling recovers that signal automatically** — the learned default directions
  above show the trees actively routing missing rows — and needs zero pipeline code, which
  also means zero train/serve skew risk from an imputer fitted on the wrong data.
- An explicit missing **flag + imputation matches native handling** (the gap here is under
  a thousandth of AUC, within noise). That is the correct fallback for models that can't
  eat `NaN`, and it keeps the signal *inspectable* — a flag can show up in importance plots.

Caution: "native handling" also means the model silently *learns from* missingness. If
missingness patterns will shift in production (a pipeline bug makes a feature 100% null),
native handling degrades silently — monitor null rates
([../10_ml_in_production/01_drift_monitoring.ipynb](../10_ml_in_production/01_drift_monitoring.ipynb)).

## 8. Native categorical handling

Trees have three classic options for a categorical with $k$ levels:

- **One-hot**: $k$ binary features. Each split isolates *one* level — with large $k$ this
  fragments the tree, wastes depth, and slows histogram building.
- **Integer codes**: one feature, but imposes a fake ordering; the tree needs several
  splits to carve out arbitrary level groups.
- **Native categorical**: LightGBM sorts levels by gradient statistics and finds the
  optimal *group* split in $O(k \log k)$ (the classic Fisher trick); XGBoost partitions
  similarly with `enable_categorical=True`. One split can send any subset of levels left.

In [ ]:
def to_onehot(X):
    return pd.get_dummies(X, columns=["device", "region_id"])


cat_rows = []
for name, transform, model in [
    ("LightGBM native categorical", lambda X: X,
     lgb.LGBMClassifier(**BASE)),
    ("LightGBM one-hot (40+3 levels)", to_onehot,
     lgb.LGBMClassifier(**BASE)),
    ("LightGBM integer codes", to_codes,
     lgb.LGBMClassifier(**BASE)),
    ("XGBoost native categorical", lambda X: X,
     xgb.XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=6, tree_method="hist",
                       enable_categorical=True, eval_metric="auc",
                       early_stopping_rounds=50, random_state=42)),
]:
    Xtr, Xva, Xte = transform(X_train), transform(X_valid), transform(X_test)
    t0 = time.perf_counter()
    if isinstance(model, lgb.LGBMClassifier):
        model.fit(Xtr, y_train, eval_X=Xva, eval_y=y_valid, eval_metric="auc",
                  callbacks=[lgb.early_stopping(50, verbose=False, first_metric_only=True)])
        best_iter = model.best_iteration_
    else:
        model.fit(Xtr, y_train, eval_set=[(Xva, y_valid)], verbose=False)
        best_iter = model.best_iteration
    cat_rows.append({
        "setup": name, "n_features": Xtr.shape[1], "best_iter": best_iter,
        "fit_s": time.perf_counter() - t0,
        "test_auc": roc_auc_score(y_test, model.predict_proba(Xte)[:, 1]),
    })
pd.DataFrame(cat_rows).round({"fit_s": 2, "test_auc": 4})

With only 43 total levels all encodings land within a few thousandths of AUC — the honest
result. The differences that *scale*:

- One-hot inflates the feature matrix (10 → 51 columns here; imagine 10,000 merchant IDs)
  and slows every histogram pass; native categorical keeps one column and finds grouped
  splits one-hot needs many levels of depth to mimic.
- Native categorical splits can **overfit high-cardinality features** (each grouping is
  chosen greedily on training gradients). LightGBM's guards — `min_data_per_group`,
  `cat_smooth`, `max_cat_threshold` — are worth knowing by name.
- For extreme cardinality (users, SKUs), the practical answer is usually **target/count
  encoding computed out-of-fold** — done wrong it is a textbook leak; see
  [02_leakage_validation.ipynb](02_leakage_validation.ipynb).

## 9. Pitfalls

Four failure modes that show up constantly in real GBT work — each demonstrated, not
just asserted.

### Pitfall 1: comparing learning rates at a fixed `n_estimators`

In [ ]:
fixed_rows = []
for lr_val in [0.3, 0.1, 0.05, 0.01]:
    model = lgb.LGBMClassifier(**{**BASE, "n_estimators": 100, "learning_rate": lr_val})
    model.fit(X_train, y_train)
    fixed_rows.append({"learning_rate": lr_val, "n_trees": 100,
                       "valid_auc_fixed100": roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])})

pitfall1 = pd.DataFrame(fixed_rows).merge(
    lr_table.rename(columns={"value": "learning_rate", "valid_auc": "valid_auc_early_stopped",
                             "best_iter": "trees_early_stopped"})
            [["learning_rate", "trees_early_stopped", "valid_auc_early_stopped"]],
    on="learning_rate")
pitfall1.round(4)

At a fixed 100 trees, *both ends* of the learning-rate range look artificially bad:
lr=0.01 is simply undertrained (it wanted ~450 trees), and lr=0.3 is worst of all because
100 trees is far past its optimum (early stopping would have quit at ~13). The "winner" of
a fixed-budget comparison is just whichever learning rate happens to match the arbitrary
tree count — here lr=0.05, whose early-stopped optimum sits near 100. Any grid search that
crosses `learning_rate` with a fixed `n_estimators` produces this artifact.
**`n_estimators` is not a free hyperparameter — it is determined by early stopping,
conditional on everything else.**

### Pitfall 2: trusting the validation score of the *selected* config

In [ ]:
top5 = search.head(5)[["value", "num_leaves", "min_child_samples", "valid_auc", "test_auc"]]
top5 = top5.rename(columns={"value": "trial"})
top5["test_rank_among_30"] = search["test_auc"].rank(ascending=False).loc[top5.index].astype(int)
print(top5.round(4).to_string(index=False))
print(f"\nvalid AUC of selected config : {best['valid_auc']:.4f}")
print(f"test  AUC of selected config : {best['test_auc']:.4f}")
print(f"best  test AUC among all 30  : {search['test_auc'].max():.4f}")
print(f"spread of test AUC in top-5 by valid: {top5['test_auc'].max() - top5['test_auc'].min():.4f}")

Two lessons in one table. First, the winner's validation score is an **optimistically
biased** estimate of its true performance *in expectation* — you took a max over 30 noisy
numbers. Here the bias happens to be invisible (the selected config's test AUC essentially
matches its validation AUC) because the validation set is large and 30 trials is few;
crank the trial count into the hundreds or shrink the selection set and it bites hard.
Report test, never the selection-set score. Second, the top trials are separated by less
than split noise, and their *ranking flips* on the test set: the config that won on valid
is not the test winner. Chasing the fourth decimal of a validation metric is fitting
noise — the deeper treatment (nested CV, repeated splits) is in
[02_leakage_validation.ipynb](02_leakage_validation.ipynb).

### Pitfall 3: trees cannot extrapolate

In [ ]:
x_tr_ex = np.linspace(0, 10, 400)
y_tr_ex = 0.8 * x_tr_ex + np.sin(2 * x_tr_ex) + np.random.default_rng(3).normal(0, 0.3, 400)
x_ex_grid = np.linspace(0, 15, 600)

gbt_ex = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.1, random_state=42, verbose=-1)
gbt_ex.fit(x_tr_ex.reshape(-1, 1), y_tr_ex)
lin_ex = LinearRegression().fit(x_tr_ex.reshape(-1, 1), y_tr_ex)

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(x_tr_ex, y_tr_ex, s=5, alpha=0.3, color="gray", label="train (x ≤ 10)")
ax.plot(x_ex_grid, gbt_ex.predict(x_ex_grid.reshape(-1, 1)), color="C3", lw=1.8, label="GBT prediction")
ax.plot(x_ex_grid, lin_ex.predict(x_ex_grid.reshape(-1, 1)), color="C0", lw=1.5, ls="--",
        label="linear regression")
ax.axvline(10, color="k", ls=":", lw=1)
ax.annotate("training range ends", xy=(10, 0.5), xytext=(10.4, -0.5), fontsize=9)
ax.set(xlabel="x", ylabel="y", title="Trees predict a constant outside the training range")
ax.legend()
plt.show()

Every tree prediction is an average of *training* targets in some leaf, so beyond the
observed range the model outputs a constant. For trending targets (revenue, demand,
anything growing), model the trend separately or predict differences — the forecasting
treatment is in
[../05_time_series/03_ml_forecasting_backtesting.ipynb](../05_time_series/03_ml_forecasting_backtesting.ipynb).

### Pitfall 4: gain importance rewards splittability, not signal

In [ ]:
rng_noise = np.random.default_rng(1)
X_train_n = X_train.copy()
X_train_n["user_id_hash"] = rng_noise.random(len(X_train))   # pure noise, ~unique per row
X_valid_n = X_valid.copy()
X_valid_n["user_id_hash"] = rng_noise.random(len(X_valid))

noisy_model = lgb.LGBMClassifier(n_estimators=300, num_leaves=127, min_child_samples=5,
                                 learning_rate=0.05, random_state=42, verbose=-1)
noisy_model.fit(X_train_n, y_train)

gain = pd.Series(noisy_model.booster_.feature_importance("gain"),
                 index=X_train_n.columns).sort_values(ascending=False)
gain_rank = pd.Series(np.arange(1, len(gain) + 1), index=gain.index, name="gain_rank")

perm = permutation_importance(noisy_model, X_valid_n.iloc[:4000], y_valid[:4000],
                              scoring="roc_auc", n_repeats=5, random_state=42)
perm_imp = pd.Series(perm.importances_mean, index=X_valid_n.columns, name="perm_importance")

importance_check = pd.concat([gain.rename("gain").round(0), gain_rank,
                              perm_imp.round(4)], axis=1).sort_values("gain", ascending=False)
print(importance_check.to_string())

`user_id_hash` is *pure noise*, yet gain importance ranks it above genuinely predictive
features — a nearly-unique continuous feature offers the greedy splitter endless
opportunities to shave training loss. Permutation importance on held-out data tells the
truth (≈ 0, sometimes negative). Never present split/gain importance as "what drives the
outcome" — the full story (permutation, SHAP, and *their* limits) is in
[05_explainability_shap.ipynb](05_explainability_shap.ipynb).

## Interview corner

**1. "Explain gradient boosting" — at three depths.**
*Executive:* build many small decision trees in sequence, each correcting the errors the
previous ones still make; sum them for the prediction. *Practitioner:* stagewise additive
model — fit a shallow tree to the current residuals (for squared loss), add it with a
shrinkage factor, repeat; regularize via learning rate, tree size, and early stopping on a
validation set. *Mathematical:* gradient descent in function space — each tree approximates
the negative functional gradient of the loss at the current model; XGBoost/LightGBM take a
Newton-style step using per-leaf first- and second-order gradient sums
($w_j^* = -G_j/(H_j+\lambda)$), with the same statistics driving split gains.

**2. XGBoost vs random forest — when and why?**
Orthogonal ensembling principles: RF averages deep, independently trained trees
(variance reduction only, parallel, nearly tuning-free, hard to overfit by adding trees);
boosting fits shallow trees sequentially (bias *and* variance reduction, needs early
stopping, *can* overfit by adding trees). Boosting usually wins a few points of accuracy on
tabular data; RF remains attractive as a low-babysitting baseline and when
variance-only averaging suits noisy targets. Know that "more trees" is safe for RF and a
hyperparameter-coupled decision for boosting.

**3. You have 30 minutes to tune — what do you touch first and why?**
In order: (1) nothing — set lr ≈ 0.05, `n_estimators` huge, early stopping on a *correct*
validation split: this fixes capacity for free; (2) `num_leaves`/`max_depth` and
`min_child_samples` — the per-tree complexity vs leaf-evidence pair, the real overfit
knobs; (3) `subsample`/`colsample_bytree` for decorrelation and speed; (4) L1/L2 last —
rarely decisive. Then a coarse random search over knobs 2–3 if time remains, expecting a
gain of a few thousandths. Saying "the validation design matters more than the grid" is
the senior answer.

**4. "The model must respect price monotonicity — what are your options?"**
(a) Native monotone constraints in XGBoost/LightGBM — one parameter, near-zero accuracy
cost, exact by construction (as demonstrated above). (b) Model architecture: fit a GAM or a
hybrid — GBT on non-price features plus an explicitly monotone (e.g., isotonic-decreasing
or parametric) price term. (c) Feature/post-processing hacks — bin price and enforce
ordering post-hoc, or project predictions to monotone via isotonic regression per segment
(approximate, can distort calibration). (d) Ask *why* monotonicity is required: if the
model feeds pricing decisions, the real issue is causal — observational price effects are
confounded, and you may need experimentation or causal methods, not just a shape
constraint. Naming option (d) is what distinguishes a senior answer.

**5. Why is LightGBM often faster than XGBoost, and what's the risk of leaf-wise growth?**
Histogram binning is common to both; LightGBM adds best-first *leaf-wise* growth (leaf
budget spent only where loss drops most), optional GOSS (keep large-gradient rows, sample
small-gradient ones), and exclusive feature bundling for sparse data. Risk: with the same
nominal capacity, leaf-wise trees grow very deep in data-sparse pockets and memorize
noise — hence `num_leaves` plus `min_child_samples` (and optionally `max_depth` as a
guard) need joint control.

**6. How does XGBoost use second-order information, and why does it matter?**
The loss is expanded to second order around the current predictions; for a fixed tree
structure the optimal leaf value is $-G_j/(H_j + \lambda)$ (gradient sum over Hessian sum
plus L2), and the split gain is expressed in the same statistics. This gives Newton-style
step sizing per leaf — better-calibrated steps for non-quadratic losses like log-loss —
and makes regularization ($\lambda$, $\gamma$) fall out of the objective rather than being
bolted on.

**7. When would you *not* use gradient boosting?**
Tiny datasets (hundreds of rows — regularized linear models or nothing); when inference is
the product (coefficients, CIs, regulatory interpretability); unstructured inputs (text,
image, audio — deep learning); targets requiring extrapolation beyond the feature range
(pitfall 3); extreme latency/memory budgets where a linear model serves in microseconds;
and any setting where the *causal* effect of an action is the question — a predictive
model, however accurate, answers a different question
([../03_causal_inference/05_uplift_cate.ipynb](../03_causal_inference/05_uplift_cate.ipynb)).

## Key takeaways

- Gradient boosting = gradient descent in function space; "fit trees to residuals" is the
  squared-loss special case, and 15 lines of NumPy reproduce sklearn exactly.
- GBTs dominate mid-size tabular ML because their inductive bias (axis-aligned thresholds,
  scale invariance, native NaN/categorical handling) matches the data — but the AUC gap
  over a well-built linear model is often modest; know what each costs to build and explain.
- `learning_rate` and `n_estimators` are one knob, not two: fix lr ∈ [0.03, 0.1], cap
  trees high, let early stopping decide. Never compare learning rates at fixed tree count.
- The overfit knobs that matter: `num_leaves`/`max_depth` (capacity per tree) and
  `min_child_samples` (evidence per leaf). Subsampling and L1/L2 are second-order.
- With early stopping in place, tuning is worth a few thousandths of AUC — data quality,
  features, and validation design are worth far more. Random search over 3–4 knobs for
  ~30 trials captures nearly all of the tuning gain.
- LightGBM grows leaf-wise (deep, asymmetric, fast, easier to overfit); XGBoost defaults
  to depth-wise (balanced). Tuned, they reach the same accuracy — pick on workflow.
- Monotone constraints encode business priors at near-zero accuracy cost — but they fix
  the *shape* of the response, not its causal magnitude; confounding survives constraints.
- Native missing handling learns default split directions and preserves MNAR signal that
  plain imputation destroys; native categorical handling finds grouped splits one-hot
  can't express compactly — but guard high-cardinality categoricals against overfitting.
- Gain importance rewards splittability (high-cardinality noise ranks high); use
  permutation importance or SHAP on held-out data for claims about signal.

## Exercises

**Exercise 1 — stochastic gradient boosting.** Modify the from-scratch loop to fit each
tree on a random 50% row subsample (like `subsample=0.5`). Compare the test-MSE-vs-rounds
curve to full-data boosting at lr = 0.1. Does subsampling help on this small, clean 1D
problem? When would you expect it to help more?

In [ ]:
# --- Solution 1 ---
def mse_curves_subsampled(lr, n_trees=300, frac=0.5, seed=11):
    r = np.random.default_rng(seed)
    X_tr, X_te = x_obs.reshape(-1, 1), x_new.reshape(-1, 1)
    F_tr = np.full(len(y_obs), y_obs.mean())
    F_te = np.full(len(y_new), y_obs.mean())
    te = []
    for _ in range(n_trees):
        idx = r.choice(len(y_obs), size=int(frac * len(y_obs)), replace=False)
        tree = DecisionTreeRegressor(max_depth=2, random_state=0)
        tree.fit(X_tr[idx], (y_obs - F_tr)[idx])          # fit on the subsample only
        F_tr = F_tr + lr * tree.predict(X_tr)             # update everyone
        F_te = F_te + lr * tree.predict(X_te)
        te.append(np.mean((y_new - F_te) ** 2))
    return np.array(te)


te_full = mse_curves(0.1)[1]
te_sub = mse_curves_subsampled(0.1)
print(f"best test MSE — full data: {te_full.min():.4f} (tree {te_full.argmin() + 1})")
print(f"best test MSE — 50% rows : {te_sub.min():.4f} (tree {te_sub.argmin() + 1})")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(te_full, label="full data per tree", color="C0")
ax.plot(te_sub, label="50% row subsample per tree", color="C2")
ax.axhline(0.16, color="k", ls="--", lw=1, label="noise floor")
ax.set(xlabel="number of trees", ylabel="test MSE", title="Stochastic gradient boosting (from scratch)")
ax.legend()
plt.show()
# Both settle near the noise floor; subsampling even edges out full-data boosting slightly
# here (decorrelated trees = mild extra regularization) while halving per-tree cost, but
# the margin is within noise. Its real value shows on larger, noisier, higher-dimensional
# data — as a variance-reduction and speed tool, not an accuracy silver bullet.

**Exercise 2 — the `learning_rate` × `num_leaves` interaction.** Using `fit_eval`, run the
2×2 grid lr ∈ {0.3, 0.03} × num_leaves ∈ {15, 255} with early stopping. Which combination
overfits hardest (largest train−valid gap)? Which is the most *economical* (valid AUC per
second of fit time)? Explain the tree-count pattern you see.

In [ ]:
# --- Solution 2 ---
grid_rows = []
for lr_val in [0.3, 0.03]:
    for nl in [15, 255]:
        row, _ = fit_eval(f"lr={lr_val}, leaves={nl}", f"{lr_val}/{nl}",
                          learning_rate=lr_val, num_leaves=nl)
        grid_rows.append(row)
grid_df = pd.DataFrame(grid_rows)[["knob", "best_iter", "train_auc", "valid_auc",
                                   "train_valid_gap", "fit_s"]]
print(grid_df.round(4).to_string(index=False))
# Patterns worth explaining out loud: 255 leaves drives the overfitting in both rows, but
# the *largest* train−valid gap belongs to lr=0.03 × 255 — the small learning rate lets
# early stopping run ~90 rounds, plenty of time to memorize — while lr=0.3 × 255 posts the
# worst valid AUC and gets chopped after ~7 trees (each huge step commits hard to noise).
# Most economical by valid-AUC-per-second: lr=0.3 × 15 leaves. Small lr × few leaves needs
# the most trees but stays honest. Valid AUC still barely moves across the grid except for
# the big-lr/big-tree corner: early stopping keeps total capacity roughly equalized.

**Exercise 3 — a second business constraint.** Marketing insists that higher `email_ctr`
must never *lower* predicted purchase probability. Add an increasing constraint on
`email_ctr` (keeping the decreasing one on `price`), verify both PDPs obey their
constraints, and measure the test-AUC cost.

In [ ]:
# --- Solution 3 ---
constraints = [{"price": -1, "email_ctr": 1}.get(c, 0) for c in FEATURES]
mono2 = lgb.LGBMClassifier(**BASE, monotone_constraints=constraints)
mono2.fit(X_train, y_train, eval_X=X_valid, eval_y=y_valid, eval_metric="auc",
          callbacks=[lgb.early_stopping(50, verbose=False, first_metric_only=True)])


def pdp_feature(model, X, col, grid):
    vals = []
    for v in grid:
        X_mod = X.copy()
        X_mod[col] = v
        vals.append(model.predict_proba(X_mod)[:, 1].mean())
    return np.array(vals)


ctr_grid = np.linspace(0.0, 0.5, 26)
pdp_price2 = pdp_feature(mono2, pdp_sample, "price", price_grid)
pdp_ctr2 = pdp_feature(mono2, pdp_sample, "email_ctr", ctr_grid)
print(f"price PDP max up-step   : {np.diff(pdp_price2).max():+.6f}  (must be <= 0)")
print(f"email_ctr PDP min step  : {np.diff(pdp_ctr2).min():+.6f}  (must be >= 0)")
auc_two = roc_auc_score(y_test, mono2.predict_proba(X_test)[:, 1])
print(f"test AUC with both constraints: {auc_two:.4f}  "
      f"(single constraint: {mono_table['test_auc'].iloc[1]:.4f}, "
      f"unconstrained: {mono_table['test_auc'].iloc[0]:.4f})")
# Both constraints hold exactly, and the AUC cost stays within noise — email_ctr's true
# effect is genuinely increasing, so the constraint again removes only noise, not signal.